# Notebook 06: Spherical Harmonics for View-Dependent Color

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/06_spherical_harmonics.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why view-dependent appearance matters in 3DGS
2. Learn the mathematical foundations of Spherical Harmonics
3. Implement SH evaluation for view-dependent color
4. Visualize SH basis functions
5. Optimize SH coefficients from observations

**Estimated Time**: 75 minutes

**Prerequisites**: Notebook 05 (Alpha Blending)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import math

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

## 1. Why View-Dependent Appearance?

### The Problem with Simple RGB

If we assign each Gaussian a fixed RGB color, we can only represent **diffuse** (Lambertian) surfaces:

- Matte surfaces that look the same from all angles
- No specular highlights
- No reflections

### Real-World Materials

Many surfaces exhibit **view-dependent** appearance:

| Material | Appearance Change |
|----------|------------------|
| Shiny metal | Specular highlights move with viewpoint |
| Water | Reflections change with angle |
| Car paint | Color shifts at grazing angles |
| Glass | Transmission vs reflection varies |

### Solution: Spherical Harmonics

Instead of storing RGB directly, we store **SH coefficients** that define a function:

$$c(\mathbf{d}) = \sum_{l=0}^{L} \sum_{m=-l}^{l} c_{l,m} \cdot Y_l^m(\mathbf{d})$$

where:
- $\mathbf{d}$ is the view direction
- $Y_l^m$ are the SH basis functions
- $c_{l,m}$ are the learned coefficients

In [ ]:
# Visualize view-dependent vs view-independent appearance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create sphere for visualization
theta = np.linspace(0, np.pi, 50)
phi = np.linspace(0, 2*np.pi, 50)
theta_grid, phi_grid = np.meshgrid(theta, phi)

# Convert to Cartesian
x = np.sin(theta_grid) * np.cos(phi_grid)
y = np.sin(theta_grid) * np.sin(phi_grid)
z = np.cos(theta_grid)

# View-independent (constant color)
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
color_constant = np.ones_like(x) * 0.5
ax1.plot_surface(x, y, z, facecolors=cm.Reds(color_constant), alpha=0.8)
ax1.set_title('View-Independent (Diffuse)\nSame color from all directions')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# View-dependent (varies with direction)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
# Simulate specular: brighter where z is high (camera looking down)
view_dir = np.array([0, 0, 1])  # Camera looking down
specular = (x * view_dir[0] + y * view_dir[1] + z * view_dir[2]) ** 4
specular = np.clip(specular, 0, 1)
ax2.plot_surface(x, y, z, facecolors=cm.Reds(0.3 + 0.7 * specular), alpha=0.8)
ax2.set_title('View-Dependent (Specular)\nBrighter towards camera')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

print("Key insight: View-dependent appearance enables realistic rendering of shiny/reflective materials")

## 2. Spherical Harmonics Basics

### What are Spherical Harmonics?

Spherical Harmonics (SH) are a set of **orthonormal basis functions** defined on the sphere.

They are analogous to Fourier series, but for functions on a sphere:
- Fourier: decompose periodic functions into sines/cosines
- SH: decompose spherical functions into spherical harmonics

### Mathematical Definition

$$Y_l^m(\theta, \phi) = K_l^m \cdot P_l^{|m|}(\cos\theta) \cdot e^{im\phi}$$

where:
- $l$ is the **degree** (0, 1, 2, ...)
- $m$ is the **order** ($-l \leq m \leq l$)
- $K_l^m$ is a normalization constant
- $P_l^m$ are Associated Legendre Polynomials

### Number of Coefficients

| Degree | # Coefficients | Total up to degree |
|--------|---------------|--------------------|
| 0 | 1 | 1 |
| 1 | 3 | 4 |
| 2 | 5 | 9 |
| 3 | 7 | 16 |

Formula: $(l+1)^2$ total coefficients for degree $l$

In [ ]:
# SH Constants (matching the official 3DGS implementation)
SH_C0 = 0.28209479177387814  # 1 / (2 * sqrt(pi))
SH_C1 = 0.4886025119029199   # sqrt(3) / (2 * sqrt(pi))

SH_C2 = [
    1.0925484305920792,   # sqrt(15) / (2 * sqrt(pi))
    -1.0925484305920792,  # -sqrt(15) / (2 * sqrt(pi))
    0.31539156525252005,  # sqrt(5) / (4 * sqrt(pi))
    -1.0925484305920792,  # -sqrt(15) / (2 * sqrt(pi))
    0.5462742152960396,   # sqrt(15) / (4 * sqrt(pi))
]

SH_C3 = [
    -0.5900435899266435,
    2.890611442640554,
    -0.4570457994644658,
    0.3731763325901154,
    -0.4570457994644658,
    1.445305721320277,
    -0.5900435899266435,
]

print("Spherical Harmonics Constants:")
print("=" * 50)
print(f"Degree 0 (DC): SH_C0 = {SH_C0:.6f}")
print(f"Degree 1: SH_C1 = {SH_C1:.6f}")
print(f"\nDegree 2 constants: {[f'{c:.4f}' for c in SH_C2]}")
print(f"\nDegree 3 constants: {[f'{c:.4f}' for c in SH_C3]}")

print("\n" + "=" * 50)
print("Number of coefficients per degree:")
for l in range(4):
    n_at_l = 2 * l + 1
    n_total = (l + 1) ** 2
    print(f"  Degree {l}: {n_at_l} new, {n_total} total (for RGB: {n_total * 3} floats)")

## 3. SH Basis Function Visualization

Let's visualize what each SH basis function looks like on a sphere.

In [ ]:
def compute_sh_basis(x, y, z, degree):
    """
    Compute SH basis function values at given points on sphere.
    
    Args:
        x, y, z: Cartesian coordinates on unit sphere
        degree: Maximum SH degree
    
    Returns:
        List of basis function values
    """
    basis = []
    
    # Degree 0 (1 function)
    Y_0_0 = SH_C0 * np.ones_like(x)
    basis.append(('Y_0^0 (constant)', Y_0_0))
    
    if degree >= 1:
        # Degree 1 (3 functions)
        Y_1_m1 = -SH_C1 * y
        Y_1_0 = SH_C1 * z
        Y_1_p1 = -SH_C1 * x
        basis.append(('Y_1^{-1} (y)', Y_1_m1))
        basis.append(('Y_1^0 (z)', Y_1_0))
        basis.append(('Y_1^1 (x)', Y_1_p1))
    
    if degree >= 2:
        # Degree 2 (5 functions)
        Y_2_m2 = SH_C2[0] * x * y
        Y_2_m1 = SH_C2[1] * y * z
        Y_2_0 = SH_C2[2] * (2*z*z - x*x - y*y)
        Y_2_p1 = SH_C2[3] * x * z
        Y_2_p2 = SH_C2[4] * (x*x - y*y)
        basis.append(('Y_2^{-2} (xy)', Y_2_m2))
        basis.append(('Y_2^{-1} (yz)', Y_2_m1))
        basis.append(('Y_2^0 (2z²-x²-y²)', Y_2_0))
        basis.append(('Y_2^1 (xz)', Y_2_p1))
        basis.append(('Y_2^2 (x²-y²)', Y_2_p2))
    
    return basis


# Create spherical sampling
resolution = 100
theta = np.linspace(0, np.pi, resolution)
phi = np.linspace(0, 2*np.pi, resolution)
theta_grid, phi_grid = np.meshgrid(theta, phi)

x = np.sin(theta_grid) * np.cos(phi_grid)
y = np.sin(theta_grid) * np.sin(phi_grid)
z = np.cos(theta_grid)

# Compute basis functions
basis_functions = compute_sh_basis(x, y, z, degree=2)

# Visualize degree 0, 1, 2
fig = plt.figure(figsize=(16, 10))

for idx, (name, Y) in enumerate(basis_functions):
    ax = fig.add_subplot(3, 3, idx + 1, projection='3d')
    
    # Normalize for coloring
    Y_norm = (Y - Y.min()) / (Y.max() - Y.min() + 1e-8)
    
    # Color by value
    colors = cm.coolwarm(Y_norm)
    
    # Plot with radius modulated by absolute value (for visualization)
    r = 0.5 + 0.5 * np.abs(Y) / (np.abs(Y).max() + 1e-8)
    ax.plot_surface(x * r, y * r, z * r, facecolors=colors, alpha=0.9, shade=False)
    
    ax.set_title(name, fontsize=10)
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])
    ax.set_box_aspect([1, 1, 1])
    ax.axis('off')

plt.suptitle('Spherical Harmonics Basis Functions (Degrees 0-2)\n'
            'Red = positive, Blue = negative, Radius ~ |value|',
            fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 4. Implementing SH Evaluation

Now let's implement the SH evaluation function used in 3DGS.

In [ ]:
def eval_sh(
    sh: torch.Tensor,
    direction: torch.Tensor,
    degree: int = 3,
) -> torch.Tensor:
    """
    Evaluate spherical harmonics at given view directions.
    
    Args:
        sh: SH coefficients [N, K, 3] where K = (degree+1)^2
        direction: View directions [N, 3] (normalized)
        degree: Maximum SH degree (0-3)
    
    Returns:
        RGB colors [N, 3]
    """
    x = direction[:, 0:1]
    y = direction[:, 1:2]
    z = direction[:, 2:3]
    
    # Degree 0 (constant)
    result = SH_C0 * sh[:, 0]
    
    if degree >= 1:
        # Degree 1 (linear)
        result = result + SH_C1 * (
            -y * sh[:, 1] + z * sh[:, 2] - x * sh[:, 3]
        )
    
    if degree >= 2:
        # Precompute products
        xx, yy, zz = x * x, y * y, z * z
        xy, yz, xz = x * y, y * z, x * z
        
        # Degree 2 (quadratic)
        result = result + SH_C2[0] * xy * sh[:, 4]
        result = result + SH_C2[1] * yz * sh[:, 5]
        result = result + SH_C2[2] * (2 * zz - xx - yy) * sh[:, 6]
        result = result + SH_C2[3] * xz * sh[:, 7]
        result = result + SH_C2[4] * (xx - yy) * sh[:, 8]
    
    if degree >= 3:
        # Degree 3 (cubic)
        result = result + SH_C3[0] * y * (3 * xx - yy) * sh[:, 9]
        result = result + SH_C3[1] * xy * z * sh[:, 10]
        result = result + SH_C3[2] * y * (4 * zz - xx - yy) * sh[:, 11]
        result = result + SH_C3[3] * z * (2 * zz - 3 * xx - 3 * yy) * sh[:, 12]
        result = result + SH_C3[4] * x * (4 * zz - xx - yy) * sh[:, 13]
        result = result + SH_C3[5] * z * (xx - yy) * sh[:, 14]
        result = result + SH_C3[6] * x * (xx - 3 * yy) * sh[:, 15]
    
    return result


# Test the implementation
print("Testing SH Evaluation:")
print("=" * 50)

N = 5  # 5 Gaussians
degree = 3
n_coeffs = (degree + 1) ** 2  # 16 for degree 3

# Random SH coefficients
sh_coeffs = torch.randn(N, n_coeffs, 3)

# Random view directions (normalized)
directions = torch.randn(N, 3)
directions = directions / directions.norm(dim=-1, keepdim=True)

# Evaluate
colors = eval_sh(sh_coeffs, directions, degree=3)

print(f"SH coefficients shape: {sh_coeffs.shape} (N={N}, K={n_coeffs}, RGB=3)")
print(f"Directions shape: {directions.shape}")
print(f"Output colors shape: {colors.shape}")
print(f"\nSample output colors:\n{colors}")

## 5. Converting Between RGB and SH

When initializing Gaussians from a point cloud with colors, we need to convert RGB to SH.

In [ ]:
def rgb_to_sh(rgb: torch.Tensor) -> torch.Tensor:
    """
    Convert RGB color to zeroth-order (DC) SH coefficient.
    
    The DC term represents the average color across all directions.
    For a diffuse (Lambertian) surface, this is the only term needed.
    
    Args:
        rgb: RGB colors [N, 3] in range [0, 1]
    
    Returns:
        SH DC coefficients [N, 3]
    """
    return (rgb - 0.5) / SH_C0


def sh_to_rgb(sh_dc: torch.Tensor) -> torch.Tensor:
    """
    Convert DC SH coefficient back to RGB.
    
    Args:
        sh_dc: DC SH coefficients [N, 3]
    
    Returns:
        RGB colors [N, 3]
    """
    return SH_C0 * sh_dc + 0.5


def initialize_sh_from_rgb(rgb: torch.Tensor, degree: int = 3) -> torch.Tensor:
    """
    Initialize full SH coefficients from RGB.
    
    DC term is set from RGB, higher-order terms are zero (diffuse).
    
    Args:
        rgb: RGB colors [N, 3]
        degree: Maximum SH degree
    
    Returns:
        SH coefficients [N, (degree+1)^2, 3]
    """
    N = rgb.shape[0]
    n_coeffs = (degree + 1) ** 2
    
    sh = torch.zeros(N, n_coeffs, 3, device=rgb.device, dtype=rgb.dtype)
    sh[:, 0, :] = rgb_to_sh(rgb)
    
    return sh


# Test conversion
print("RGB ↔ SH Conversion:")
print("=" * 50)

# Original RGB colors
rgb_original = torch.tensor([
    [1.0, 0.0, 0.0],  # Red
    [0.0, 1.0, 0.0],  # Green
    [0.0, 0.0, 1.0],  # Blue
    [0.5, 0.5, 0.5],  # Gray
])

# Convert to SH
sh_dc = rgb_to_sh(rgb_original)

# Convert back
rgb_recovered = sh_to_rgb(sh_dc)

print(f"Original RGB:\n{rgb_original}")
print(f"\nSH DC coefficients:\n{sh_dc}")
print(f"\nRecovered RGB:\n{rgb_recovered}")
print(f"\nReconstruction error: {(rgb_original - rgb_recovered).abs().max().item():.10f}")

# Test full initialization
sh_full = initialize_sh_from_rgb(rgb_original, degree=3)
print(f"\nFull SH shape: {sh_full.shape} (4 Gaussians, 16 coeffs, 3 channels)")
print(f"Non-zero coefficients: {(sh_full.abs() > 1e-6).sum().item()} (should be 12 = 4 * 3 for DC only)")

## 6. View-Dependent Color Demo

Let's see how SH coefficients create view-dependent color.

In [ ]:
def visualize_sh_on_sphere(sh_coeffs, degree=3, resolution=50, title="SH Color on Sphere"):
    """
    Visualize how SH coefficients produce color on a sphere.
    
    Args:
        sh_coeffs: [1, K, 3] or [K, 3] SH coefficients for one point
        degree: SH degree
        resolution: Sampling resolution
        title: Plot title
    """
    if sh_coeffs.dim() == 2:
        sh_coeffs = sh_coeffs.unsqueeze(0)
    
    # Create spherical sampling
    theta = torch.linspace(0, math.pi, resolution)
    phi = torch.linspace(0, 2 * math.pi, resolution)
    theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')
    
    # Convert to Cartesian (view directions)
    x = torch.sin(theta_grid) * torch.cos(phi_grid)
    y = torch.sin(theta_grid) * torch.sin(phi_grid)
    z = torch.cos(theta_grid)
    
    directions = torch.stack([x.flatten(), y.flatten(), z.flatten()], dim=-1)
    
    # Expand SH coefficients for all directions
    N = directions.shape[0]
    sh_expanded = sh_coeffs.expand(N, -1, -1)
    
    # Evaluate SH
    colors = eval_sh(sh_expanded, directions, degree=degree)
    
    # Clamp to valid range and reshape
    colors = torch.clamp(colors + 0.5, 0, 1)  # Add 0.5 offset
    colors = colors.reshape(resolution, resolution, 3)
    
    # Plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot_surface(
        x.numpy(), y.numpy(), z.numpy(),
        facecolors=colors.numpy(),
        shade=False,
        alpha=1.0
    )
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.set_box_aspect([1, 1, 1])
    
    plt.tight_layout()
    return fig, ax


# Create different SH configurations
fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': '3d'})

configs = [
    ('DC only (Diffuse Red)', lambda: create_sh_config('dc_red')),
    ('DC only (Diffuse Blue)', lambda: create_sh_config('dc_blue')),
    ('Degree 1: z-gradient', lambda: create_sh_config('z_gradient')),
    ('Degree 1: x-gradient', lambda: create_sh_config('x_gradient')),
    ('Degree 2: xy term', lambda: create_sh_config('xy_term')),
    ('Mixed: Specular-like', lambda: create_sh_config('specular')),
]

def create_sh_config(config_type):
    """Create SH coefficients for different configurations."""
    sh = torch.zeros(16, 3)
    
    if config_type == 'dc_red':
        sh[0] = rgb_to_sh(torch.tensor([1.0, 0.2, 0.2]))
    elif config_type == 'dc_blue':
        sh[0] = rgb_to_sh(torch.tensor([0.2, 0.2, 1.0]))
    elif config_type == 'z_gradient':
        sh[0] = rgb_to_sh(torch.tensor([0.5, 0.5, 0.5]))
        sh[2] = torch.tensor([1.0, 0.0, 0.0])  # Y_1^0 (z-dependent)
    elif config_type == 'x_gradient':
        sh[0] = rgb_to_sh(torch.tensor([0.5, 0.5, 0.5]))
        sh[3] = torch.tensor([0.0, 0.0, 1.0])  # Y_1^1 (x-dependent)
    elif config_type == 'xy_term':
        sh[0] = rgb_to_sh(torch.tensor([0.5, 0.5, 0.5]))
        sh[4] = torch.tensor([0.8, 0.0, 0.8])  # Y_2^{-2} (xy)
    elif config_type == 'specular':
        sh[0] = rgb_to_sh(torch.tensor([0.3, 0.3, 0.8]))
        sh[2] = torch.tensor([0.5, 0.5, 0.5])  # z-gradient for highlight
        sh[6] = torch.tensor([0.3, 0.3, 0.3])  # Degree 2 term
    
    return sh


# Create spherical sampling
resolution = 50
theta = torch.linspace(0, math.pi, resolution)
phi = torch.linspace(0, 2 * math.pi, resolution)
theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')

x_sphere = torch.sin(theta_grid) * torch.cos(phi_grid)
y_sphere = torch.sin(theta_grid) * torch.sin(phi_grid)
z_sphere = torch.cos(theta_grid)

directions = torch.stack([x_sphere.flatten(), y_sphere.flatten(), z_sphere.flatten()], dim=-1)
N = directions.shape[0]

for idx, (title, config_fn) in enumerate(configs):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    sh_coeffs = config_fn().unsqueeze(0).expand(N, -1, -1)
    colors = eval_sh(sh_coeffs, directions, degree=3)
    colors = torch.clamp(colors + 0.5, 0, 1).reshape(resolution, resolution, 3)
    
    ax.plot_surface(
        x_sphere.numpy(), y_sphere.numpy(), z_sphere.numpy(),
        facecolors=colors.numpy(),
        shade=False,
        alpha=1.0
    )
    ax.set_title(title)
    ax.set_box_aspect([1, 1, 1])
    ax.axis('off')

plt.suptitle('View-Dependent Color from Different SH Configurations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. View Direction Computation

In 3DGS, the view direction is computed from the Gaussian position to the camera center.

In [ ]:
def get_view_direction(
    gaussian_positions: torch.Tensor,
    camera_center: torch.Tensor,
) -> torch.Tensor:
    """
    Compute view direction from Gaussians to camera.
    
    Args:
        gaussian_positions: [N, 3] world positions of Gaussians
        camera_center: [3] world position of camera
    
    Returns:
        [N, 3] normalized view directions
    """
    direction = camera_center.unsqueeze(0) - gaussian_positions
    direction = direction / (direction.norm(dim=-1, keepdim=True) + 1e-8)
    return direction


# Demo: multiple Gaussians, different camera positions
print("View Direction Computation:")
print("=" * 50)

# Gaussian at origin
gaussian_pos = torch.tensor([[0., 0., 0.]])

# Different camera positions
camera_positions = torch.tensor([
    [0., 0., 5.],    # Looking from +Z
    [0., 0., -5.],   # Looking from -Z
    [5., 0., 0.],    # Looking from +X
    [0., 5., 0.],    # Looking from +Y
    [3., 3., 3.],    # Diagonal
])

print("Gaussian position: [0, 0, 0]")
print("\nCamera position → View direction (normalized)")
for cam_pos in camera_positions:
    view_dir = get_view_direction(gaussian_pos, cam_pos)
    print(f"  {cam_pos.tolist()} → {view_dir[0].tolist()}")


# Visualize
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Gaussian at origin
ax.scatter([0], [0], [0], s=200, c='red', marker='o', label='Gaussian')

# Camera positions and view directions
for i, cam_pos in enumerate(camera_positions):
    view_dir = get_view_direction(gaussian_pos, cam_pos)[0]
    
    # Camera
    ax.scatter(*cam_pos.tolist(), s=100, c='blue', marker='^')
    ax.text(*cam_pos.tolist(), f'  C{i}', fontsize=10)
    
    # View direction arrow (from Gaussian toward camera)
    ax.quiver(0, 0, 0, view_dir[0]*2, view_dir[1]*2, view_dir[2]*2,
              color='green', alpha=0.6, arrow_length_ratio=0.1)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('View Directions from Gaussian to Cameras')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Optimizing SH Coefficients

Let's optimize SH coefficients to match observed colors from different viewpoints.

In [ ]:
# Create ground truth: a point with known view-dependent color
torch.manual_seed(42)

# Ground truth SH coefficients (degree 2)
sh_gt = torch.zeros(1, 9, 3)
sh_gt[0, 0] = rgb_to_sh(torch.tensor([0.6, 0.3, 0.3]))  # DC: reddish
sh_gt[0, 2] = torch.tensor([0.0, 0.5, 0.0])  # Y_1^0: green toward +z
sh_gt[0, 3] = torch.tensor([0.0, 0.0, 0.3])  # Y_1^1: blue toward +x

# Generate training data: observed colors from different directions
n_observations = 50
train_directions = torch.randn(n_observations, 3)
train_directions = train_directions / train_directions.norm(dim=-1, keepdim=True)

# Get observed colors
sh_gt_expanded = sh_gt.expand(n_observations, -1, -1)
train_colors = eval_sh(sh_gt_expanded, train_directions, degree=2) + 0.5
train_colors = torch.clamp(train_colors, 0, 1)

print(f"Training data: {n_observations} observations from different viewpoints")
print(f"Direction range: [{train_directions.min():.3f}, {train_directions.max():.3f}]")
print(f"Color range: [{train_colors.min():.3f}, {train_colors.max():.3f}]")

In [ ]:
# Optimize SH coefficients to fit observations
degree = 2
n_coeffs = (degree + 1) ** 2

# Initialize with small random values
sh_learned = torch.zeros(1, n_coeffs, 3, requires_grad=True)

optimizer = torch.optim.Adam([sh_learned], lr=0.1)

losses = []
n_iterations = 200

for i in range(n_iterations):
    optimizer.zero_grad()
    
    # Predict colors
    sh_expanded = sh_learned.expand(n_observations, -1, -1)
    pred_colors = eval_sh(sh_expanded, train_directions, degree=degree) + 0.5
    
    # Loss
    loss = F.mse_loss(pred_colors, train_colors)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if i % 50 == 0:
        print(f"Iter {i:3d}: Loss = {loss.item():.6f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
# Compare learned vs ground truth
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Loss curve
axes[0, 0].semilogy(losses)
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Loss (log scale)')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

# SH coefficients comparison
axes[0, 1].bar(np.arange(n_coeffs) - 0.2, sh_gt[0, :, 0].detach().numpy(), 
              width=0.4, label='GT', color='blue', alpha=0.7)
axes[0, 1].bar(np.arange(n_coeffs) + 0.2, sh_learned[0, :, 0].detach().numpy(),
              width=0.4, label='Learned', color='red', alpha=0.7)
axes[0, 1].set_xlabel('Coefficient Index')
axes[0, 1].set_ylabel('Value (Red channel)')
axes[0, 1].set_title('SH Coefficients (Red Channel)')
axes[0, 1].legend()
axes[0, 1].set_xticks(range(n_coeffs))

# Prediction accuracy
sh_expanded = sh_learned.expand(n_observations, -1, -1)
final_pred = eval_sh(sh_expanded, train_directions, degree=degree) + 0.5

for i, channel in enumerate(['Red', 'Green', 'Blue']):
    axes[0, 2].scatter(train_colors[:, i].detach(), final_pred[:, i].detach(),
                       alpha=0.5, label=channel, s=30)
axes[0, 2].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[0, 2].set_xlabel('Ground Truth')
axes[0, 2].set_ylabel('Predicted')
axes[0, 2].set_title('Prediction vs Ground Truth')
axes[0, 2].legend()
axes[0, 2].set_aspect('equal')

# Visualize on sphere: GT, Learned, Difference
resolution = 40
theta = torch.linspace(0, math.pi, resolution)
phi = torch.linspace(0, 2 * math.pi, resolution)
theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')

x_sphere = torch.sin(theta_grid) * torch.cos(phi_grid)
y_sphere = torch.sin(theta_grid) * torch.sin(phi_grid)
z_sphere = torch.cos(theta_grid)

test_directions = torch.stack([x_sphere.flatten(), y_sphere.flatten(), z_sphere.flatten()], dim=-1)
N_test = test_directions.shape[0]

# Ground truth colors on sphere
gt_colors = eval_sh(sh_gt.expand(N_test, -1, -1), test_directions, degree=2) + 0.5
gt_colors = torch.clamp(gt_colors, 0, 1).reshape(resolution, resolution, 3)

# Learned colors on sphere
learned_colors = eval_sh(sh_learned.expand(N_test, -1, -1).detach(), test_directions, degree=2) + 0.5
learned_colors = torch.clamp(learned_colors, 0, 1).reshape(resolution, resolution, 3)

for idx, (title, colors) in enumerate([('Ground Truth', gt_colors), 
                                        ('Learned', learned_colors)]):
    ax = fig.add_subplot(2, 3, 4 + idx, projection='3d')
    ax.plot_surface(x_sphere.numpy(), y_sphere.numpy(), z_sphere.numpy(),
                   facecolors=colors.detach().numpy(), shade=False, alpha=1.0)
    ax.set_title(title)
    ax.set_box_aspect([1, 1, 1])
    ax.axis('off')

# Difference
ax = fig.add_subplot(2, 3, 6, projection='3d')
diff = (gt_colors - learned_colors).abs().mean(dim=-1)
ax.plot_surface(x_sphere.numpy(), y_sphere.numpy(), z_sphere.numpy(),
               facecolors=cm.hot(diff.detach().numpy()), shade=False, alpha=1.0)
ax.set_title('Absolute Difference')
ax.set_box_aspect([1, 1, 1])
ax.axis('off')

plt.tight_layout()
plt.show()

print(f"\nMax SH coefficient difference: {(sh_gt[:, :n_coeffs] - sh_learned).abs().max().item():.4f}")

## 9. SH Degree Trade-offs

Higher SH degree = more view-dependent detail, but more parameters.

In [ ]:
# Compare different SH degrees
print("SH Degree Comparison:")
print("=" * 60)

degrees = [0, 1, 2, 3]

for d in degrees:
    n_coeffs = (d + 1) ** 2
    params_per_gaussian = n_coeffs * 3  # RGB
    
    # For 1M Gaussians
    n_gaussians = 1_000_000
    total_mb = (n_gaussians * params_per_gaussian * 4) / (1024 * 1024)  # float32
    
    # Frequency content
    if d == 0:
        freq = "Constant (diffuse only)"
    elif d == 1:
        freq = "Linear variation"
    elif d == 2:
        freq = "Quadratic (soft reflections)"
    else:
        freq = "Cubic (sharper highlights)"
    
    print(f"\nDegree {d}:")
    print(f"  Coefficients per color: {n_coeffs}")
    print(f"  Parameters per Gaussian: {params_per_gaussian}")
    print(f"  Memory for 1M Gaussians: {total_mb:.1f} MB")
    print(f"  Frequency content: {freq}")

In [ ]:
# Visualize what each degree can represent
fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw={'projection': '3d'})

# Create a complex ground truth pattern
resolution = 60
theta = torch.linspace(0, math.pi, resolution)
phi = torch.linspace(0, 2 * math.pi, resolution)
theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')

x = torch.sin(theta_grid) * torch.cos(phi_grid)
y = torch.sin(theta_grid) * torch.sin(phi_grid)
z = torch.cos(theta_grid)

# Ground truth: complex pattern with specular highlight
highlight_dir = torch.tensor([0.5, 0.3, 0.8])
highlight_dir = highlight_dir / highlight_dir.norm()

# Compute dot product with highlight direction
dot = x * highlight_dir[0] + y * highlight_dir[1] + z * highlight_dir[2]
specular = torch.clamp(dot, 0, 1) ** 16  # Sharp specular

# Base color + specular
gt_colors = torch.stack([
    0.3 + 0.7 * specular,  # Red with specular
    0.2 + 0.2 * z.clamp(0, 1) + 0.6 * specular,  # Green varies with z
    0.4 + 0.4 * specular,  # Blue with specular
], dim=-1)

# Fit different degree SH to this pattern
directions = torch.stack([x.flatten(), y.flatten(), z.flatten()], dim=-1)
gt_flat = gt_colors.reshape(-1, 3)

for deg_idx, degree in enumerate([0, 1, 2, 3]):
    n_coeffs = (degree + 1) ** 2
    N = directions.shape[0]
    
    # Fit SH by least squares (simplified)
    sh_fit = torch.zeros(1, 16, 3)  # Use degree 3 storage
    sh_fit[0, 0] = rgb_to_sh(gt_flat.mean(dim=0))  # DC from average
    
    # For higher degrees, use gradient descent
    if degree > 0:
        sh_param = torch.zeros(1, n_coeffs, 3, requires_grad=True)
        sh_param.data[0, 0] = rgb_to_sh(gt_flat.mean(dim=0))
        
        opt = torch.optim.Adam([sh_param], lr=0.05)
        for _ in range(100):
            opt.zero_grad()
            pred = eval_sh(sh_param.expand(N, -1, -1), directions, degree=degree) + 0.5
            loss = F.mse_loss(pred, gt_flat)
            loss.backward()
            opt.step()
        
        sh_fit[0, :n_coeffs] = sh_param.detach()
    
    # Evaluate
    pred_colors = eval_sh(sh_fit.expand(N, -1, -1), directions, degree=degree) + 0.5
    pred_colors = torch.clamp(pred_colors, 0, 1).reshape(resolution, resolution, 3)
    
    # Plot
    ax = axes[deg_idx]
    ax.plot_surface(x.numpy(), y.numpy(), z.numpy(),
                   facecolors=pred_colors.detach().numpy(), shade=False)
    
    error = F.mse_loss(pred_colors.flatten(), gt_colors.flatten()).item()
    ax.set_title(f'Degree {degree}\nMSE: {error:.4f}')
    ax.set_box_aspect([1, 1, 1])
    ax.axis('off')

plt.suptitle('Fitting Complex Pattern with Different SH Degrees\n(Higher degree = better fit)',
            fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

## 10. Integration with 3DGS Rendering

Here's how SH fits into the full rendering pipeline.

In [ ]:
def render_with_sh(
    gaussian_positions: torch.Tensor,  # [N, 3]
    gaussian_sh: torch.Tensor,         # [N, K, 3]
    gaussian_opacities: torch.Tensor,  # [N]
    gaussian_2d_covs: torch.Tensor,    # [N, 2, 2]
    gaussian_2d_means: torch.Tensor,   # [N, 2]
    camera_center: torch.Tensor,       # [3]
    height: int,
    width: int,
    sh_degree: int = 3,
) -> torch.Tensor:
    """
    Simplified 2D rendering with SH-based view-dependent color.
    
    This shows the integration of SH evaluation into rendering.
    """
    N = gaussian_positions.shape[0]
    
    # Step 1: Compute view direction for each Gaussian
    view_dirs = get_view_direction(gaussian_positions, camera_center)
    
    # Step 2: Evaluate SH to get colors
    colors = eval_sh(gaussian_sh, view_dirs, degree=sh_degree) + 0.5
    colors = torch.clamp(colors, 0, 1)
    
    # Step 3: Render using alpha blending (simplified)
    y = torch.arange(height, dtype=torch.float32)
    x = torch.arange(width, dtype=torch.float32)
    y_grid, x_grid = torch.meshgrid(y, x, indexing='ij')
    
    image = torch.zeros(height, width, 3)
    transmittance = torch.ones(height, width)
    
    for i in range(N):
        mean = gaussian_2d_means[i]
        cov = gaussian_2d_covs[i]
        cov_inv = torch.linalg.inv(cov)
        opacity = gaussian_opacities[i]
        color = colors[i]
        
        dx = x_grid - mean[0]
        dy = y_grid - mean[1]
        
        mahal = (cov_inv[0, 0] * dx * dx +
                (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
                cov_inv[1, 1] * dy * dy)
        
        gaussian_val = torch.exp(-0.5 * mahal)
        alpha = gaussian_val * opacity
        weight = transmittance * alpha
        
        for c in range(3):
            image[:, :, c] += weight * color[c]
        
        transmittance = transmittance * (1 - alpha)
    
    # Background
    for c in range(3):
        image[:, :, c] += transmittance * 1.0
    
    return torch.clamp(image, 0, 1)


# Demo: same Gaussian, different camera positions
N = 1
gaussian_pos = torch.tensor([[0., 0., 0.]])
gaussian_2d_mean = torch.tensor([[50., 50.]])
gaussian_2d_cov = torch.tensor([[[400., 0.], [0., 400.]]])
gaussian_opacity = torch.tensor([0.9])

# SH with view-dependent component
gaussian_sh = torch.zeros(N, 16, 3)
gaussian_sh[0, 0] = rgb_to_sh(torch.tensor([0.5, 0.3, 0.3]))
gaussian_sh[0, 2] = torch.tensor([0.3, 0.3, 0.0])  # z-dependent
gaussian_sh[0, 3] = torch.tensor([0.0, 0.0, 0.4])  # x-dependent

# Different camera positions
camera_positions = [
    torch.tensor([0., 0., 5.]),
    torch.tensor([5., 0., 0.]),
    torch.tensor([0., 5., 0.]),
    torch.tensor([3., 3., 3.]),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, cam_pos in enumerate(camera_positions):
    image = render_with_sh(
        gaussian_pos, gaussian_sh, gaussian_opacity,
        gaussian_2d_cov, gaussian_2d_mean, cam_pos,
        height=100, width=100, sh_degree=3
    )
    
    axes[idx].imshow(image.numpy())
    axes[idx].set_title(f'Camera at {cam_pos.tolist()}')
    axes[idx].axis('off')

plt.suptitle('Same Gaussian, Different View Directions → Different Colors',
            fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The Gaussian appears different from each viewpoint due to SH-encoded view-dependent color!")

## 11. Summary: Spherical Harmonics in 3DGS

### Key Equations

| Component | Formula | Description |
|-----------|---------|-------------|
| SH Evaluation | $c(\mathbf{d}) = \sum_{l,m} c_{l,m} Y_l^m(\mathbf{d})$ | Color from direction |
| View Direction | $\mathbf{d} = \frac{\text{cam} - \mu}{\|\text{cam} - \mu\|}$ | Gaussian to camera |
| DC Term | $Y_0^0 = \frac{1}{2\sqrt{\pi}}$ | Constant (average) color |
| RGB ↔ SH | $\text{sh} = (\text{rgb} - 0.5) / C_0$ | Conversion |

### SH Degrees in 3DGS

| Degree | # Coeffs | Parameters/Gaussian | Use Case |
|--------|----------|--------------------|-----------|
| 0 | 1 | 3 | Diffuse only |
| 1 | 4 | 12 | Smooth gradients |
| 2 | 9 | 27 | Soft reflections |
| 3 | 16 | 48 | Sharp highlights |

### Key Points

1. **SH enables view-dependent appearance** - essential for realistic rendering
2. **SH coefficients are learnable parameters** - optimized during training
3. **Degree 3 is standard** - good balance of quality and memory
4. **View direction is computed per-Gaussian** - from Gaussian to camera

---

## Key Takeaways

1. SH provides a compact way to represent view-dependent color
2. Higher degree = more detail, but more parameters
3. DC term represents average color (diffuse component)
4. Higher-order terms capture view-dependent effects

---

## Next Steps

In the next notebook, we'll learn about **Adaptive Density Control**:

**[07_adaptive_density_control.ipynb](./07_adaptive_density_control.ipynb)** - Gaussian Splitting, Cloning, and Pruning